# Section 5.11, Figure 1

Dynamic subnetwork prediction and generalization on the sklearn digits dataset.

## Run Notes
- Run the notebook from top to bottom.
- This notebook uses sklearn digits, so it does not download an external dataset.
- The default training run is intentionally long (`epochs = 200`) because the program model is trained from scratch.
- The transformer mask-program model has several diagnostics that are plotted after training.

## Setup

In [ ]:
import copy
import math
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset


def signed_kaiming_constant_(tensor, a=0, mode="fan_in", nonlinearity="relu", k=0.5):
    """Initialize frozen mask-vocabulary weights with signed Kaiming-style samples."""
    fan = nn.init._calculate_correct_fan(tensor, mode)
    gain = nn.init.calculate_gain(nonlinearity, a)
    std = gain / math.sqrt(fan)
    if k != 0:
        std *= 1 / math.sqrt(k)

    with torch.no_grad():
        tensor.uniform_(-std, std)
        return tensor


class GetSubnet(autograd.Function):
    """Straight-through top-k mask used by edge-popup style score tensors."""

    @staticmethod
    def forward(ctx, scores, k=0.5):
        out = scores.clone()
        _, idx = scores.flatten().sort()
        cutoff = int((1 - k) * scores.numel())
        flat_out = out.flatten()
        flat_out[idx[:cutoff]] = 0
        flat_out[idx[cutoff:]] = 1
        return out

    @staticmethod
    def backward(ctx, grad):
        return grad, None

## Experiment and Figure

In [ ]:
# Experiment outline:
# 1. Load and standardize the sklearn digits dataset.
# 2. Train a transformer-controlled mask-program model that dynamically selects
#    learned mask tokens for each input.
# 3. Train a classical MLP with comparable realized depth.
# 4. Plot accuracy/loss curves, program-length diagnostics, token usage,
#    common generated programs by class, and the final confusion matrix.

# ============================================================
# Device / seed
# ============================================================
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else "cpu"
)
print("device:", device)


# ============================================================
# Dataset
# ============================================================
digits = load_digits()
X = digits.data.astype(np.float32)
y = digits.target.astype(np.int64)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.2,
    random_state=SEED,
    stratify=y_trainval,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

batch_size = 128
train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
    batch_size=batch_size,
    shuffle=True
)
val_loader = DataLoader(
    TensorDataset(torch.tensor(X_val), torch.tensor(y_val)),
    batch_size=batch_size,
    shuffle=False
)
test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test), torch.tensor(y_test)),
    batch_size=batch_size,
    shuffle=False
)


# ============================================================
# Helpers
# ============================================================
def smooth(x, k=5):
    x = np.asarray(x, dtype=float)
    if len(x) < k:
        return x
    return np.convolve(x, np.ones(k) / k, mode="valid")


def empirical_entropy(counts, eps=1e-12):
    counts = np.asarray(counts, dtype=float)
    s = counts.sum()
    if s <= 0:
        return 0.0
    p = counts / s
    return float(-(p * np.log(p + eps)).sum())


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def token_sequence_to_string(seq):
    return " ".join(f"M{int(t)}" for t in seq)


def avg_from_hist(hist):
    hist = np.asarray(hist, dtype=float)
    return float(np.dot(np.arange(len(hist)), hist) / max(hist.sum(), 1.0))


# ============================================================
# Classical baseline
# ============================================================
class ClassicalMLP(nn.Module):
    def __init__(self, in_size=64, hidden_size=128, out_size=10, num_hidden_layers=4):
        super().__init__()
        layers = [nn.Linear(in_size, hidden_size), nn.ReLU()]

        for _ in range(num_hidden_layers - 1):
            layers += [nn.Linear(hidden_size, hidden_size), nn.ReLU()]

        layers += [nn.Linear(hidden_size, out_size)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ============================================================
# Learned mask vocabulary
# ============================================================
class LearnedMaskVocabularyLayer(nn.Module):
    def __init__(self, features, num_masks, mask_embed_dim=64, k=0.5):
        super().__init__()
        self.features = features
        self.num_masks = num_masks
        self.mask_embed_dim = mask_embed_dim
        self.k = k

        self.weight = nn.Parameter(torch.empty(features, features), requires_grad=False)
        signed_kaiming_constant_(self.weight)

        self.bias = nn.Parameter(torch.zeros(features), requires_grad=False)

        self.mask_embeddings = nn.Embedding(num_masks, mask_embed_dim)

        self.weight_score_decoder = nn.Sequential(
            nn.Linear(mask_embed_dim, mask_embed_dim),
            nn.ReLU(),
            nn.Linear(mask_embed_dim, features * (2 * features))
        )

        self.bias_score_decoder = nn.Sequential(
            nn.Linear(mask_embed_dim, mask_embed_dim),
            nn.ReLU(),
            nn.Linear(mask_embed_dim, 2 * features)
        )

    def all_decoded_scores(self):
        token_ids = torch.arange(self.num_masks, device=self.weight.device)
        emb = self.mask_embeddings(token_ids)

        w_scores = self.weight_score_decoder(emb)
        w_scores = w_scores.view(self.num_masks, self.features, 2 * self.features)

        b_scores = self.bias_score_decoder(emb)
        b_scores = b_scores.view(self.num_masks, 2, self.features)

        return w_scores, b_scores

    def all_masked_params(self):
        weight_scores, bias_scores = self.all_decoded_scores()

        Ws = []
        bs = []

        for m in range(self.num_masks):
            mask = GetSubnet.apply(weight_scores[m].abs(), self.k)
            Wm = self.weight * mask[:, :self.features]

            bmask = GetSubnet.apply(bias_scores[m].abs(), self.k)
            bm = self.bias * bmask[0, :self.features]

            Ws.append(Wm)
            bs.append(bm)

        W_all = torch.stack(Ws, dim=0)
        b_all = torch.stack(bs, dim=0)
        return W_all, b_all

    def apply_cached(self, h, token_weights, W_all, b_all):
        """
        h:             [B, features]
        token_weights: [B, num_masks]
        W_all:         [num_masks, features, features]
        b_all:         [num_masks, features]
        """
        W_batch = torch.einsum("bm,mij->bij", token_weights, W_all)
        b_batch = torch.einsum("bm,mj->bj", token_weights, b_all)
        return torch.einsum("bij,bj->bi", W_batch, h) + b_batch

    def decoded_mask_diversity_loss(self):
        weight_scores, _ = self.all_decoded_scores()
        M = weight_scores.shape[0]
        flat = weight_scores.reshape(M, -1)
        flat = F.normalize(flat, dim=1)
        G = flat @ flat.T
        eye = torch.eye(M, device=G.device)
        offdiag = G * (1.0 - eye)
        return (offdiag ** 2).sum() / max(M * (M - 1), 1)


# ============================================================
# Transformer policy with separate halt head
# ============================================================
class MaskSequenceTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=128,
        nhead=4,
        num_layers=2,
        ff_mult=4,
        max_len=512
    ):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=ff_mult * d_model,
            batch_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.token_out = nn.Linear(d_model, vocab_size)
        self.halt_out = nn.Linear(d_model, 1)

    def forward(self, token_ids, context_embed):
        B, T = token_ids.shape

        pos = torch.arange(T, device=token_ids.device).unsqueeze(0).expand(B, T)

        x = self.token_emb(token_ids) + self.pos_emb(pos)

        # Replace START-token embedding with context.
        x[:, 0, :] = context_embed + self.pos_emb(pos)[:, 0, :]

        causal_mask = torch.triu(
            torch.ones(T, T, device=token_ids.device, dtype=torch.bool),
            diagonal=1
        )

        z = self.transformer(x, mask=causal_mask)

        token_logits = self.token_out(z)
        halt_logits = self.halt_out(z).squeeze(-1)

        return token_logits, halt_logits


def straight_through_gumbel_one_hot(logits, tau=1.0, eps=1e-20):
    g = -torch.log(-torch.log(torch.rand_like(logits) + eps) + eps)
    probs = torch.softmax((logits + g) / tau, dim=-1)
    idx = probs.argmax(dim=-1)
    y_hard = F.one_hot(idx, num_classes=logits.shape[-1]).float()
    y = y_hard.detach() - probs.detach() + probs
    return y, idx, probs


# ============================================================
# Program model with differentiable halting mass
# ============================================================
class TransformerProgramLinearSubnetIO(nn.Module):
    def __init__(
        self,
        in_size=64,
        hidden_size=128,
        out_size=10,
        num_masks=8,
        max_steps=128,
        k=0.5,
        d_model=128,
        nhead=4,
        num_layers=2,
        mask_embed_dim=64,
        min_steps=1,
        halt_threshold=1e-4,
        max_active_steps_for_transformer=None,
        freeze_io=False
    ):
        super().__init__()

        self.in_size = in_size
        self.hidden_size = hidden_size
        self.out_size = out_size
        self.num_masks = num_masks
        self.max_steps = max_steps
        self.min_steps = min_steps
        self.halt_threshold = halt_threshold

        # If max_steps is enormous, this prevents positional embedding blow-up.
        # Usually keep this equal to max_steps + 1 unless you are testing massive lengths.
        if max_active_steps_for_transformer is None:
            max_active_steps_for_transformer = max_steps + 1
        self.max_active_steps_for_transformer = max_active_steps_for_transformer

        self.act = nn.ReLU()

        self.in_layer = nn.Linear(in_size, hidden_size, bias=True)

        self.vocab_layer = LearnedMaskVocabularyLayer(
            hidden_size,
            num_masks,
            mask_embed_dim=mask_embed_dim,
            k=k
        )

        self.out_layer = nn.Linear(hidden_size, out_size, bias=True)

        if freeze_io:
            for p in self.in_layer.parameters():
                p.requires_grad = False
            for p in self.out_layer.parameters():
                p.requires_grad = False

        self.start_id = num_masks
        self.vocab_size = num_masks + 1

        self.context_proj = nn.Linear(hidden_size, d_model)

        self.policy = MaskSequenceTransformer(
            vocab_size=self.vocab_size,
            d_model=d_model,
            nhead=nhead,
            num_layers=num_layers,
            max_len=max_active_steps_for_transformer
        )

        # Diagnostics / aux losses.
        self.token_usage = None
        self.seq_len_hist = None
        self.soft_len_mean = None
        self.route_entropy = None
        self.halt_prob_mean = None
        self.generated_sequences = None

        self.token_usage_entropy_loss = None
        self.per_sample_usage_loss = None
        self.mask_diversity_loss = None
        self.step_entropy_loss = None
        self.repeat_token_loss = None
        self.length_loss = None
        self.halt_conf_loss = None

    def forward(self, x, tau=1.0, use_conf_halt_target=True):
        h = self.act(self.in_layer(x))
        initial_context = self.context_proj(h)

        B = h.size(0)
        device_ = h.device

        # Cache decoded masks once per forward pass.
        W_all, b_all = self.vocab_layer.all_masked_params()

        active = torch.ones(B, dtype=torch.bool, device=device_)
        active_mass = torch.ones(B, device=device_)

        hard_seq_len = torch.zeros(B, dtype=torch.long, device=device_)
        soft_seq_len = torch.zeros(B, device=device_)

        token_counts = torch.zeros(self.num_masks, device=device_)
        per_sample_mask_mass = torch.zeros(B, self.num_masks, device=device_)

        generated_tokens = [[] for _ in range(B)]
        prefix_tokens = torch.full(
            (B, 1),
            self.start_id,
            dtype=torch.long,
            device=device_
        )

        entropy_accum = 0.0
        entropy_batches = 0

        halt_prob_accum = 0.0
        halt_prob_batches = 0

        all_mask_probs = []
        all_step_entropies = []
        repeat_losses = []
        halt_conf_losses = []

        prev_mask_probs_by_sample = torch.zeros(B, self.num_masks, device=device_)
        has_prev_by_sample = torch.zeros(B, dtype=torch.bool, device=device_)

        for step in range(self.max_steps):
            if not torch.any(active):
                break

            # Safety if the transformer positional embedding is intentionally smaller
            # than max_steps. This is not a curriculum; it is just a context-window limit.
            if prefix_tokens.shape[1] >= self.max_active_steps_for_transformer:
                break

            active_idx = torch.where(active)[0]
            prefix_active = prefix_tokens[active_idx]
            context_active = initial_context[active_idx]

            token_logits_full, halt_logits_full = self.policy(prefix_active, context_active)

            logits_next = token_logits_full[:, -1, :]
            halt_logit = halt_logits_full[:, -1]

            # Do not generate START after initialization.
            logits_next[:, self.start_id] = -1e9

            if self.training:
                route_st, choice, probs = straight_through_gumbel_one_hot(logits_next, tau=tau)
            else:
                probs = torch.softmax(logits_next, dim=-1)
                choice = logits_next.argmax(dim=1)
                route_st = F.one_hot(choice, num_classes=self.vocab_size).float()

            mask_probs = probs[:, :self.num_masks]
            mask_weights = route_st[:, :self.num_masks]

            # Halt probability is separate from the mask token choice.
            halt_prob = torch.sigmoid(halt_logit)

            # Force at least min_steps before halting is permitted.
            if step < self.min_steps:
                halt_prob = halt_prob * 0.0

            step_mass = active_mass[active_idx]

            # Expected compute cost.
            soft_seq_len[active_idx] += step_mass

            # Diagnostics.
            entropy = -(probs * torch.log(probs + 1e-12)).sum(dim=1).mean()
            step_entropy = -(mask_probs * torch.log(mask_probs + 1e-12)).sum(dim=1).mean()
            all_step_entropies.append(step_entropy)
            all_mask_probs.append(mask_probs)

            entropy_accum += float(entropy.detach().cpu())
            entropy_batches += 1

            halt_prob_accum += float(halt_prob.mean().detach().cpu())
            halt_prob_batches += 1

            # Consecutive-repeat penalty.
            has_prev_active = has_prev_by_sample[active_idx]
            if torch.any(has_prev_active):
                prev = prev_mask_probs_by_sample[active_idx[has_prev_active]]
                curr = mask_probs[has_prev_active]
                repeat_losses.append((prev * curr).sum(dim=1).mean())

            # Optional confidence-informed halting target.
            # This is intentionally soft and detached.
            if use_conf_halt_target:
                with torch.no_grad():
                    interim_logits = self.out_layer(h[active_idx])
                    interim_conf = torch.softmax(interim_logits, dim=-1).max(dim=1).values

                    # Confidence near 1 says "you may halt."
                    # Confidence near chance says "probably continue."
                    chance = 1.0 / self.out_size
                    target_halt = ((interim_conf - chance) / (1.0 - chance + 1e-12)).clamp(
                        0.0,
                        1.0,
                    )

                halt_conf_losses.append(
                    F.binary_cross_entropy(halt_prob, target_halt)
                )

            # Apply selected mask-program step.
            h_old = h[active_idx]
            h_new = self.vocab_layer.apply_cached(h_old, mask_weights, W_all, b_all)
            h_new = self.act(h_new)

            # Soft residual update:
            # if active_mass is small, the step contributes weakly.
            # This is the main stabilization trick for giant max_steps.
            alpha = step_mass.unsqueeze(1).clamp(0.0, 1.0)
            h_updated = (1.0 - alpha) * h_old + alpha * h_new

            h_next = h.clone()
            h_next[active_idx] = h_updated
            h = h_next

            # Hard diagnostics.
            hard_seq_len[active_idx] += 1
            per_sample_mask_mass[active_idx] += step_mass.unsqueeze(1) * mask_probs

            for loc, glob in enumerate(active_idx.tolist()):
                tok = int(choice[loc].item())
                generated_tokens[glob].append(tok)

            choice_detached = choice.detach()
            for m in range(self.num_masks):
                token_counts[m] += (choice_detached == m).sum()

            prev_mask_probs_by_sample[active_idx] = mask_weights.detach() - mask_probs.detach() + mask_probs
            has_prev_by_sample[active_idx] = True

            # Update differentiable active mass.
            active_mass_next = active_mass.clone()
            active_mass_next[active_idx] = step_mass * (1.0 - halt_prob)
            active_mass = active_mass_next

            active = active_mass > self.halt_threshold

            # Append chosen token for next autoregressive step.
            append_col = torch.full((B, 1), self.start_id, dtype=torch.long, device=device_)
            append_col[active_idx, 0] = choice
            prefix_tokens = torch.cat([prefix_tokens, append_col], dim=1)

        self.generated_sequences = generated_tokens
        self.seq_len_hist = np.bincount(
            hard_seq_len.detach().cpu().numpy(),
            minlength=self.max_steps + 1
        )
        self.token_usage = token_counts.detach().cpu().numpy()
        self.soft_len_mean = float(soft_seq_len.mean().detach().cpu())
        std_len = soft_seq_len.std(unbiased=False)

        self.length_variance_bonus = -std_len / max(float(self.max_steps), 1.0)
        self.route_entropy = entropy_accum / max(entropy_batches, 1)
        self.halt_prob_mean = halt_prob_accum / max(halt_prob_batches, 1)

        if len(all_mask_probs) > 0:
            P = torch.cat(all_mask_probs, dim=0)
            mean_p = P.mean(dim=0)

            # Negative entropy: adding this to the loss rewards global usage diversity.
            self.token_usage_entropy_loss = torch.sum(mean_p * torch.log(mean_p + 1e-12))
        else:
            self.token_usage_entropy_loss = torch.tensor(0.0, device=device_)

        p_sample = per_sample_mask_mass / (per_sample_mask_mass.sum(dim=1, keepdim=True) + 1e-12)
        self.per_sample_usage_loss = torch.sum(
            p_sample * torch.log(p_sample + 1e-12),
            dim=1
        ).mean()

        if len(repeat_losses) > 0:
            self.repeat_token_loss = torch.stack(repeat_losses).mean()
        else:
            self.repeat_token_loss = torch.tensor(0.0, device=device_)

        self.mask_diversity_loss = self.vocab_layer.decoded_mask_diversity_loss()

        if len(all_step_entropies) > 0:
            # Negative entropy: adding this to the loss rewards per-step token diversity.
            self.step_entropy_loss = torch.stack(all_step_entropies).mean()
        else:
            self.step_entropy_loss = torch.tensor(0.0, device=device_)

        # Expected compute cost. This is the important giant-length term.
        soft_len = soft_seq_len.mean()

        target_min = 8.0
        target_max = 40.0

        too_short = F.relu(target_min - soft_len).pow(2)
        too_long = F.relu(soft_len - target_max).pow(2)

        self.length_loss = (too_short + too_long) / (self.max_steps ** 2)
        self.soft_len_tensor = soft_len

        if len(halt_conf_losses) > 0:
            self.halt_conf_loss = torch.stack(halt_conf_losses).mean()
        else:
            self.halt_conf_loss = torch.tensor(0.0, device=device_)

        return self.out_layer(h)


# ============================================================
# Evaluation
# ============================================================
@torch.no_grad()
def evaluate_classical(model, loader):
    model.eval()
    total = 0
    correct = 0
    total_loss = 0.0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        loss = F.cross_entropy(logits, y)
        preds = logits.argmax(dim=1)

        total += x.size(0)
        correct += (preds == y).sum().item()
        total_loss += loss.item() * x.size(0)

    return {"acc": correct / total, "loss": total_loss / total}


@torch.no_grad()
def evaluate_program(model, loader, collect_stats=False):
    model.eval()

    total = 0
    correct = 0
    total_loss = 0.0

    token_usage = None
    seq_len_hist = None
    entropies = []
    halt_probs = []
    soft_lens = []

    y_true = []
    y_pred = []
    all_sequences = []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x, tau=1.0)
        loss = F.cross_entropy(logits, y)
        preds = logits.argmax(dim=1)

        total += x.size(0)
        correct += (preds == y).sum().item()
        total_loss += loss.item() * x.size(0)

        if collect_stats:
            if token_usage is None:
                token_usage = model.token_usage.copy()
                seq_len_hist = model.seq_len_hist.copy()
            else:
                token_usage += model.token_usage
                seq_len_hist += model.seq_len_hist

            entropies.append(model.route_entropy)
            halt_probs.append(model.halt_prob_mean)
            soft_lens.append(model.soft_len_mean)

            all_sequences.extend(model.generated_sequences)
            y_true.append(y.cpu().numpy())
            y_pred.append(preds.cpu().numpy())

    out = {
        "acc": correct / total,
        "loss": total_loss / total
    }

    if collect_stats:
        out["token_usage"] = token_usage
        out["seq_len_hist"] = seq_len_hist
        out["route_entropy"] = float(np.mean(entropies)) if entropies else 0.0
        out["halt_prob_mean"] = float(np.mean(halt_probs)) if halt_probs else 0.0
        out["soft_len_mean"] = float(np.mean(soft_lens)) if soft_lens else 0.0
        out["y_true"] = np.concatenate(y_true) if len(y_true) else None
        out["y_pred"] = np.concatenate(y_pred) if len(y_pred) else None
        out["sequences"] = all_sequences

    return out


# ============================================================
# Training
# ============================================================
def train_classical(model, train_loader, val_loader, epochs=40, lr=1e-3):
    model.to(device)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)

    history = {
        "train_acc": [],
        "val_acc": [],
        "train_loss": [],
        "val_loss": []
    }

    best_state = None
    best_val_acc = -1.0

    for ep in range(epochs):
        model.train()

        total_loss = 0.0
        n_seen = 0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            opt.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            opt.step()

            total_loss += loss.item() * x.size(0)
            n_seen += x.size(0)

        train_stats = evaluate_classical(model, train_loader)
        val_stats = evaluate_classical(model, val_loader)

        history["train_acc"].append(train_stats["acc"])
        history["val_acc"].append(val_stats["acc"])
        history["train_loss"].append(total_loss / n_seen)
        history["val_loss"].append(val_stats["loss"])

        if val_stats["acc"] > best_val_acc:
            best_val_acc = val_stats["acc"]
            best_state = copy.deepcopy(model.state_dict())

        if ep == 0 or (ep + 1) % 10 == 0 or ep == epochs - 1:
            print(
                f"[CLASSICAL] epoch {ep+1:03d} | "
                f"train acc={train_stats['acc']:.4f} | "
                f"val acc={val_stats['acc']:.4f}"
            )

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history


def train_program(
    model,
    train_loader,
    val_loader,
    epochs=40,
    lr=1e-3,
    lambda_mask_div=0.01,
    lambda_usage=0.02,
    lambda_per_sample=0.02,
    lambda_step_entropy=0.005,
    lambda_repeat=0.05,
    lambda_length=0.05,
    lambda_halt_conf=0.001,
    tau_start=1.0,
    tau_end=0.5,
    lambda_len_var=0.01
):
    model.to(device)

    opt = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr
    )

    history = {
        "train_acc": [],
        "val_acc": [],
        "train_loss": [],
        "val_loss": [],
        "tau": [],
        "usage_reg": [],
        "per_sample_reg": [],
        "step_entropy_reg": [],
        "repeat_reg": [],
        "mask_div_reg": [],
        "length_reg": [],
        "halt_conf_reg": [],
        "hard_avg_len": [],
        "soft_avg_len": [],
        "halt_prob": []
    }

    best_state = None
    best_val_acc = -1.0

    for ep in range(epochs):
        model.train()

        if epochs <= 1:
            tau = tau_end
        else:
            frac = ep / (epochs - 1)
            tau = tau_start * (tau_end / tau_start) ** frac

        lambda_repeat_ep = lambda_repeat * (ep / max(epochs - 1, 1))

        running_loss = 0.0
        running_usage = 0.0
        running_per_sample = 0.0
        running_step_entropy = 0.0
        running_repeat = 0.0
        running_mask_div = 0.0
        running_length = 0.0
        running_halt_conf = 0.0

        n_seen = 0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            opt.zero_grad()

            logits = model(x, tau=tau)

            cls_loss = F.cross_entropy(logits, y)

            loss = (
                cls_loss
                + lambda_usage * model.token_usage_entropy_loss
                + lambda_per_sample * model.per_sample_usage_loss
                + lambda_mask_div * model.mask_diversity_loss
                + lambda_step_entropy * model.step_entropy_loss
                + lambda_repeat_ep * model.repeat_token_loss
                + lambda_length * model.length_loss
                + lambda_halt_conf * model.halt_conf_loss
                + lambda_len_var * model.length_variance_bonus
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

            bs = x.size(0)

            running_loss += loss.item() * bs
            running_usage += float(model.token_usage_entropy_loss.detach().cpu()) * bs
            running_per_sample += float(model.per_sample_usage_loss.detach().cpu()) * bs
            running_step_entropy += float(model.step_entropy_loss.detach().cpu()) * bs
            running_repeat += float(model.repeat_token_loss.detach().cpu()) * bs
            running_mask_div += float(model.mask_diversity_loss.detach().cpu()) * bs
            running_length += float(model.length_loss.detach().cpu()) * bs
            running_halt_conf += float(model.halt_conf_loss.detach().cpu()) * bs

            n_seen += bs

        train_stats = evaluate_program(model, train_loader, collect_stats=False)
        val_stats = evaluate_program(model, val_loader, collect_stats=True)

        hard_avg_len = avg_from_hist(val_stats["seq_len_hist"])
        soft_avg_len = val_stats["soft_len_mean"]
        halt_prob = val_stats["halt_prob_mean"]

        history["train_acc"].append(train_stats["acc"])
        history["val_acc"].append(val_stats["acc"])
        history["train_loss"].append(running_loss / n_seen)
        history["val_loss"].append(val_stats["loss"])
        history["tau"].append(tau)

        history["usage_reg"].append(running_usage / n_seen)
        history["per_sample_reg"].append(running_per_sample / n_seen)
        history["step_entropy_reg"].append(running_step_entropy / n_seen)
        history["repeat_reg"].append(running_repeat / n_seen)
        history["mask_div_reg"].append(running_mask_div / n_seen)
        history["length_reg"].append(running_length / n_seen)
        history["halt_conf_reg"].append(running_halt_conf / n_seen)

        history["hard_avg_len"].append(hard_avg_len)
        history["soft_avg_len"].append(soft_avg_len)
        history["halt_prob"].append(halt_prob)

        if val_stats["acc"] > best_val_acc:
            best_val_acc = val_stats["acc"]
            best_state = copy.deepcopy(model.state_dict())

        if ep == 0 or (ep + 1) % 10 == 0 or ep == epochs - 1:
            print(
                f"[POLICY] epoch {ep+1:03d} | "
                f"tau={tau:.3f} | "
                f"train acc={train_stats['acc']:.4f} | "
                f"val acc={val_stats['acc']:.4f} | "
                f"hard_len={hard_avg_len:.2f} | "
                f"soft_len={soft_avg_len:.2f} | "
                f"halt={halt_prob:.3f} | "
                f"len_reg={running_length/n_seen:.4f}"
            )

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history


# ============================================================
# Main comparison experiment
# ============================================================
hidden_size = 64

# You can now make this large.
# Start with 64 or 128 before going truly huge.
max_steps = 64

# If you want max_steps=512 or 1024, consider keeping this lower at first,
# e.g. 129 or 257, because the transformer context window is expensive.
max_active_steps_for_transformer = max_steps + 1

num_masks = 8
epochs = 200

program_model = TransformerProgramLinearSubnetIO(
    in_size=64,
    hidden_size=hidden_size,
    out_size=10,
    num_masks=num_masks,
    max_steps=max_steps,
    k=0.5,
    d_model=64,
    nhead=4,
    num_layers=2,
    mask_embed_dim=32,
    min_steps=1,
    halt_threshold=1e-3,
    max_active_steps_for_transformer=max_active_steps_for_transformer,
    freeze_io=False
)

program_model, program_hist = train_program(
    program_model,
    train_loader,
    val_loader,
    epochs=epochs,
    lr=1e-3,
    lambda_usage=0.02,
    lambda_per_sample=0.02,
    lambda_mask_div=0.01,
    lambda_step_entropy=0.005,
    lambda_repeat=0.05,
    lambda_length=0.05,
    lambda_halt_conf=0.001,
    lambda_len_var=0.01,
    tau_start=1.0,
    tau_end=0.5
)



program_test = evaluate_program(program_model, test_loader, collect_stats=True)

classical = ClassicalMLP(
    in_size=64,
    hidden_size=hidden_size,
    out_size=10,
    num_hidden_layers=int(avg_from_hist(program_test["seq_len_hist"]))
)

classical, classical_hist = train_classical(
    classical,
    train_loader,
    val_loader,
    epochs=epochs,
    lr=1e-3
)
classical_test = evaluate_classical(classical, test_loader)

print("\nParameter counts:")
print("Classical:", count_parameters(classical))
print("Program  :", count_parameters(program_model))

print("\nFinal test accuracy:")
print("Classical:", classical_test["acc"])
print("Program  :", program_test["acc"])

print("\nFinal program length diagnostics:")
print("hard avg len:", avg_from_hist(program_test["seq_len_hist"]))
print("soft avg len:", program_test["soft_len_mean"])
print("halt prob mean:", program_test["halt_prob_mean"])


# ============================================================
# Comparison charts
# ============================================================
plt.figure(figsize=(7, 5))
plt.plot(smooth(classical_hist["val_acc"], 5), linewidth=2, label="Classical val")
plt.plot(smooth(program_hist["val_acc"], 5), linewidth=2, label="Program val")
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title("Validation accuracy comparison")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(smooth(classical_hist["train_acc"], 5), linewidth=2, label="Classical train")
plt.plot(smooth(program_hist["train_acc"], 5), linewidth=2, label="Program train")
plt.xlabel("Epoch")
plt.ylabel("Training accuracy")
plt.title("Training accuracy comparison")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(smooth(classical_hist["val_loss"], 5), linewidth=2, label="Classical val")
plt.plot(smooth(program_hist["val_loss"], 5), linewidth=2, label="Program val")
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Validation loss comparison")
plt.yscale("log")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(program_hist["hard_avg_len"], linewidth=2, label="Hard realized length")
plt.plot(program_hist["soft_avg_len"], linewidth=2, label="Soft expected length")
plt.xlabel("Epoch")
plt.ylabel("Length")
plt.title("Program length during training")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# ============================================================
# Program-model-specific diagnostics
# ============================================================
token_usage = program_test["token_usage"].astype(float)
token_probs = token_usage / max(token_usage.sum(), 1.0)
order = np.argsort(-token_probs)

plt.figure(figsize=(7, 4.5))
plt.bar(np.arange(len(token_probs)), token_probs[order])
plt.xticks(
    np.arange(len(token_probs)),
    [f"M{i}" for i in order],
    rotation=0
)
plt.xlabel("Mask token, sorted by usage")
plt.ylabel("Usage probability")
plt.title("Program model: token usage distribution")
plt.tight_layout()
plt.show()


# ============================================================
# Most common generated programs by class
# ============================================================
top_k_per_class = 3

y_true = program_test["y_true"]
sequences = program_test["sequences"]

class_to_counter = {}
for cls in sorted(np.unique(y_true)):
    cls_sequences = [
        tuple(seq)
        for seq, yy in zip(sequences, y_true)
        if yy == cls
    ]
    class_to_counter[int(cls)] = Counter(cls_sequences)

# Collect every program that appears in the top-k list for at least one class.
top_programs = []
seen = set()

for cls in sorted(class_to_counter):
    for seq, cnt in class_to_counter[cls].most_common(top_k_per_class):
        if seq not in seen:
            seen.add(seq)
            top_programs.append(seq)

# Build class-by-program frequency matrix.
classes = sorted(class_to_counter)
freq = np.zeros((len(classes), len(top_programs)), dtype=float)

for i, cls in enumerate(classes):
    total = sum(class_to_counter[cls].values())
    for j, seq in enumerate(top_programs):
        freq[i, j] = class_to_counter[cls][seq] / max(total, 1)

program_labels = [
    token_sequence_to_string(seq)
    for seq in top_programs
]

plt.figure(figsize=(max(9, 0.55 * len(top_programs)), 5.5))
im = plt.imshow(freq, aspect="auto")
plt.colorbar(im, label="Within-class frequency")
plt.yticks(np.arange(len(classes)), [str(c) for c in classes])
plt.xticks(np.arange(len(top_programs)), program_labels, rotation=75, ha="right")
plt.xlabel("Generated program")
plt.ylabel("True class")
plt.title(f"Most common generated programs by class, top {top_k_per_class} per class")
plt.tight_layout()
plt.show()


# Optional: print the same information in a more readable form.
print("\nMost common generated programs by true class:")
for cls in classes:
    print(f"\nClass {cls}:")
    total = sum(class_to_counter[cls].values())
    for seq, cnt in class_to_counter[cls].most_common(top_k_per_class):
        pct = 100.0 * cnt / max(total, 1)
        seq_str = token_sequence_to_string(seq)
        print(f"  {cnt:>4} ({pct:5.1f}%) : {seq_str}")


# ============================================================
# Confusion matrix
# ============================================================
cm = confusion_matrix(program_test["y_true"], program_test["y_pred"])
_, ax = plt.subplots(figsize=(6, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax, colorbar=False)
ax.set_title("Program model confusion matrix")
plt.tight_layout()
plt.show()


print("\nProgram diagnostics:")
print("token entropy:", empirical_entropy(token_usage))
print("fraction of tokens used:", float((token_usage > 0).sum() / len(token_usage)))
print("route entropy:", program_test["route_entropy"])
print("hard avg len:", avg_from_hist(program_test["seq_len_hist"]))
print("soft avg len:", program_test["soft_len_mean"])
print("halt prob mean:", program_test["halt_prob_mean"])